# Week 05 — Android Networking with Retrofit
## LeafGuard AI: Understanding HTTP and Client-Server Communication

Retrofit is an Android networking library, but the underlying ideas are general web concepts. This notebook uses Python to explore requests, uploads, JSON parsing, async callbacks, and error handling.

**Learning outcomes:**
- Understand HTTP methods, status codes, and headers
- Simulate multipart image uploads
- Parse JSON payloads like Gson does on Android
- Explore asynchronous callback patterns with threads
- Practice defensive error handling strategies

---


## Cell 1: Build a Simple HTTP Request Model

Before using Retrofit annotations, it helps to understand the shape of a plain HTTP request.


In [ ]:
import json

request_model = {
    'method': 'POST',
    'url': 'https://api.leafguard.example/predict',
    'headers': {
        'Accept': 'application/json',
        'Authorization': '******',
        'User-Agent': 'LeafGuard-Android/1.0'
    },
    'body_type': 'multipart/form-data'
}

print(json.dumps(request_model, indent=2))


## Cell 2: Compare HTTP Status Code Families

Retrofit surfaces HTTP success and failure, but you still need to understand what different response classes mean.


In [ ]:
status_examples = {
    200: 'Success: prediction returned',
    400: 'Client error: malformed request or bad parameters',
    401: 'Authentication required',
    404: 'Endpoint not found',
    500: 'Server error: backend crashed or misconfigured',
}

for code_value, meaning in status_examples.items():
    family = f'{code_value // 100}xx'
    print(f'{code_value} ({family}) -> {meaning}')


## Cell 3: Simulate Multipart File Upload Encoding

Image uploads use boundaries to separate form fields and file bytes.


In [ ]:
        import uuid

        boundary = '----LeafGuardBoundary' + uuid.uuid4().hex[:8]
        image_bytes = b'FAKE_IMAGE_BYTES_FOR_NOTEBOOK'

        multipart_body = (
            f'--{boundary}
'
            'Content-Disposition: form-data; name="image"; filename="leaf.jpg"
'
            'Content-Type: image/jpeg

'
        ).encode('utf-8') + image_bytes + f'
--{boundary}--
'.encode('utf-8')

        print(f'Boundary: {boundary}')
        print(f'Payload size: {len(multipart_body)} bytes')
        print(multipart_body[:140])


## Cell 4: Parse Server JSON Like Gson Would

On Android, Gson maps JSON into Java/Kotlin models. In Python, the `json` module plays the same educational role.


In [ ]:
response_json = """
{
  "success": true,
  "prediction": {
    "disease": "Tomato Late Blight",
    "confidence": 0.884,
    "isHealthy": false
  },
  "requestId": "req-2048"
}
"""

payload = json.loads(response_json)
print('Disease:', payload['prediction']['disease'])
print('Confidence:', round(payload['prediction']['confidence'] * 100, 2), '%')
print('Request ID:', payload['requestId'])


## Cell 5: Simulate an Async Callback Pattern

Network calls should not block the UI thread. This example uses a background thread and a callback to mimic asynchronous work.


In [ ]:
import threading
import time

def fake_network_call(callback):
    def worker():
        time.sleep(0.2)
        callback({'status': 200, 'message': 'Prediction complete'})
    thread = threading.Thread(target=worker)
    thread.start()
    return thread

def on_result(result):
    print('Callback received:', result)

t = fake_network_call(on_result)
t.join()
print('Main thread stayed responsive while waiting.')


## Cell 6: Error Handling Strategy Table

Production apps need structured responses to timeouts, empty bodies, and server failures.


In [ ]:
error_strategy = {
    'timeout': 'Retry with backoff and show a friendly loading/error message',
    'http_400': 'Inspect request fields and validate user input',
    'http_500': 'Log request ID and surface a generic server error',
    'invalid_json': 'Treat as backend contract mismatch and fail safely',
    'no_internet': 'Offer offline mode or cached guidance if available',
}

for error_key, strategy in error_strategy.items():
    print(f'{error_key:<12} -> {strategy}')


## Cell 7: Optional `requests` Availability Check

Retrofit is a high-level client. In Python, `requests` is a comparable convenience library for learning and testing.


In [ ]:
try:
    import requests
    print(f'requests version: {requests.__version__}')
    print('You can use requests for real API testing in Python scripts.')
except ImportError:
    print('requests is not installed. Install with: pip install requests')


## Summary and Quick Quiz

**Key takeaways:**
- Retrofit is built on HTTP ideas such as methods, headers, bodies, and status codes.
- Multipart uploads are structured text + binary payloads.
- Async execution and defensive error handling are essential in mobile networking.

**Quick quiz:**
1. Why should image uploads use multipart form data instead of plain JSON?
2. What is the difference between a 4xx and 5xx HTTP error?
3. Why must networking stay off the UI thread?
4. What fields would you include in a prediction response model?


## Parallel Kotlin Track — Week 05: Retrofit Networking in Kotlin

The HTTP, multipart, and JSON concepts above are identical in the Kotlin twin. The API **contract**
does not change: `@Multipart @POST("predict")` to the same FastAPI backend.

### Java vs Kotlin: the Retrofit interface

Java (`network/ApiService.java`):

```java
public interface ApiService {
    @Multipart
    @POST("predict")
    Call<PredictionResponse> uploadImage(@Part MultipartBody.Part image);
}
```

Kotlin (`network/ApiService.kt`):

```kotlin
interface ApiService {
    @Multipart
    @POST("predict")
    fun uploadImage(@Part image: MultipartBody.Part): Call<PredictionResponse>
}
```

### The response model: POJO → data class

```kotlin
data class PredictionResponse(
    @SerializedName("disease") var disease: String? = null,
    @SerializedName("confidence") var confidence: Float = 0f,
    @SerializedName("symptoms") var symptoms: String? = null,
    @SerializedName("treatment") var treatment: String? = null,
    @SerializedName("prevention") var prevention: String? = null
)
```

One `data class` replaces ~60 lines of Java getters/setters. The `@SerializedName` values keep the
JSON parsing byte-for-byte identical.

### The singleton client: static class → object

```kotlin
object RetrofitClient {
    private const val DEFAULT_BASE_URL = "http://10.0.2.2:8000/"
    // same 30s timeouts, same HttpLoggingInterceptor levels as Java
}
```

Kotlin's `object` declaration gives a thread-safe singleton for free — no private constructor or
static methods needed. `10.0.2.2` still means "the host machine" from the Android emulator.

### Try this
Run the FastAPI backend and point **both** apps at it. Upload the same leaf image from each.

**Expected output:** identical JSON responses — the backend cannot tell which language sent the request.

**✅ Checkpoint:** which part of the HTTP request encodes the multipart form part name, and why must both tracks use the same one?

**⚠️ Common mistake:** calling `execute()` instead of `enqueue()` — Kotlin does not save you from `NetworkOnMainThreadException`.

**🔑 Key point:** the network contract lives in HTTP, not in the programming language; both tracks share the exact same backend.